In [1]:
import pandas as pd
import json
import sqlite3

In [2]:
businesses = []
with open('../data/business.json', 'r', encoding='utf-8') as f:
    for line in f:
        businesses.append(json.loads(line))

df = pd.DataFrame(businesses)
print(df.shape)

(150346, 14)


In [3]:
ca_cities = df[df['state'] == 'CA']['city'].value_counts()
print(ca_cities)

city
Santa Barbara                       3829
Goleta                               798
Carpinteria                          298
Isla Vista                            94
Montecito                             93
Summerland                            41
Truckee                               11
Santa Barbara                          5
Reno                                   3
Santa Barbra                           2
South Lake Tahoe                       1
Santa Barbara,                         1
Spring Hill                            1
Aliso Viejo                            1
SANTA BARBARA AP                       1
Costa Mesa                             1
Santa Barbara & Ventura Counties       1
SANTA BARBARA                          1
Valencia                               1
Santa  Barbara                         1
Salinas                                1
Carpinteria                            1
Real Goleta                            1
Cerritos                               1
Santa Clara

In [4]:
df_sb = df[df['city'] == 'Santa Barbara']
df_restaurants = df_sb[df_sb['categories'].str.contains('Restaurants', na=False)].copy()
print(df_restaurants.shape)

(767, 14)


In [5]:
cols_to_drop = [col for col in df_restaurants.columns 
                if df_restaurants[col].apply(lambda x: isinstance(x, dict)).any()]

print("Dropping columns:", cols_to_drop)

df_restaurants_clean = df_restaurants.drop(columns=cols_to_drop)

conn = sqlite3.connect('../data/yelp.db')
df_restaurants_clean.to_sql('restaurants', conn, if_exists='replace', index=False)
print(f"Loaded {len(df_restaurants_clean)} restaurants into database")

Dropping columns: ['attributes', 'hours']
Loaded 767 restaurants into database


In [6]:
query = '''
SELECT *
FROM restaurants
LIMIT 20
'''

result = pd.read_sql_query(query, conn)
result

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories
0,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,34.414445,-119.690672,4.0,389,1,"Food, Restaurants, Salad, Coffee & Tea, Breakf..."
1,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,34.408715,-119.685019,4.0,2404,1,"Live/Raw Food, Restaurants, Seafood, Beer Bar,..."
2,ifjluUv4VASwmFqEp8cWlQ,Marty's Pizza,2733 De La Vina St,Santa Barbara,CA,93105,34.436236,-119.726147,4.0,64,1,"Pizza, Restaurants"
3,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,34.423302,-119.705471,4.5,116,1,"Themed Cafes, Cafes, Pets, Arts & Entertainmen..."
4,Hqz96v1ymucUKNzIWfEKXw,Subway,"1936 State St, Ste B",Santa Barbara,CA,93101,34.430822,-119.714156,3.0,5,0,"Restaurants, Delis, Sandwiches, Fast Food"
5,JWFpjvCc_nkNDVtMPx1ZGg,Bogo SB,1114 State St,Santa Barbara,CA,93101,34.423043,-119.703286,4.5,12,1,"Food, Food Delivery Services, Burgers, Pizza, ..."
6,9L-MR0arflwFMF9szEBOOg,California Wine Festival,,Santa Barbara,CA,93101,34.414466,-119.685190,4.5,7,1,"Wineries, Food, Restaurants, Arts & Entertainm..."
7,aYMpjij5ShtEoZueMrQPRw,The Mill,406 E Haley St,Santa Barbara,CA,93101,34.421119,-119.690633,4.0,5,1,"Breweries, Shopping, Food Court, Food, Venues ..."
8,GgcvRnt5_z3NEC0D6vNncQ,Cafe Buenos Aires,1316 State St,Santa Barbara,CA,93101,34.424829,-119.706032,3.5,56,0,"Restaurants, Argentine"
9,N-CBZKO37y_WSUN7ZMNGkg,Taqueria El Pastorcito,4427 Hollister Ave,Santa Barbara,CA,93110,34.439706,-119.770838,4.0,66,0,"Restaurants, Mexican"


In [7]:
# What star rating distribution separates open vs. closed restaurants?

In [8]:
query = '''
SELECT
    is_open,
    stars,
    COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY is_open, stars
ORDER BY is_open, stars
'''

result = pd.read_sql_query(query, conn)
result

,is_open,stars,total_restaurants
0,0,1.5,1
1,0,2.0,7
2,0,2.5,19
3,0,3.0,29
4,0,3.5,104
5,0,4.0,104
6,0,4.5,49
7,0,5.0,8
8,1,1.5,3
9,1,2.0,9


In [9]:
# What proportion of open restaurants are rated 4.0+ vs what proportion of closed restaurants are rated 4.0+?
# Open restaurants are 20 percentage points more likely to be rated 4.0+ than closed ones (69% vs 49%).

In [10]:
query = """
WITH percentage AS (
    SELECT
    is_open,
    stars,
    COUNT(*) AS total_restaurants,
    SUM(COUNT(*)) OVER (PARTITION BY is_open) AS totals
FROM restaurants
GROUP BY is_open, stars
ORDER BY is_open, stars
)
SELECT *,
    100* (CAST(total_restaurants AS float) /totals) AS proportion
FROM percentage
WHERE stars >= 4.0
"""

result = pd.read_sql_query(query, conn)
result

,is_open,stars,total_restaurants,totals,proportion
0,0,4.0,104,321,32.398754
1,0,4.5,49,321,15.264798
2,0,5.0,8,321,2.492212
3,1,4.0,160,446,35.874439
4,1,4.5,118,446,26.457399
5,1,5.0,29,446,6.502242


In [11]:
# Which cuisine categories have the highest average ratings?
# Desserts, American (New), and Seafood are the top-rated cuisines in Santa Barbara
# Fast Food is a significant outlier at 3.02 — nearly a full point below everything else
# Sample size varies widely, so conclusions should be weighted accordingly

In [34]:
cuisines_to_keep = ['American (New)', 'Breakfast & Brunch', 'Mexican', 'Sandwiches', 
                    'American (Traditional)', 'Seafood', 'Coffee & Tea', 'Pizza', 
                    'Burgers', 'Delis', 'Salad', 'Italian', 'Bakeries', 
                    'Fast Food', 'Desserts', 'Japanese', 'Vegetarian']

exploded = df_restaurants.copy()
exploded['categories'] = exploded['categories'].str.split(", ")
exploded = exploded.explode('categories')

filtered = exploded[exploded['categories'].isin(cuisines_to_keep)]
print(filtered['categories'].value_counts().head(20))

avg = filtered.groupby('categories').agg({'stars': ['mean', 'count']}).sort_values(by=('stars', 'mean'), ascending=False)
print(avg)

categories
American (New)            151
Breakfast & Brunch        119
Mexican                   118
Sandwiches                103
American (Traditional)     95
Seafood                    78
Coffee & Tea               74
Pizza                      71
Burgers                    57
Salad                      56
Delis                      56
Italian                    55
Bakeries                   49
Fast Food                  41
Desserts                   31
Japanese                   30
Vegetarian                 28
Name: count, dtype: int64
                           stars      
                            mean count
categories                            
Desserts                4.112903    31
American (New)          3.953642   151
Seafood                 3.929487    78
Vegetarian              3.910714    28
Bakeries                3.908163    49
Delis                   3.883929    56
Salad                   3.883929    56
Coffee & Tea            3.878378    74
Mexican                 